# Landing — trust price history (CSV archive)

Source: `data/uk_investment_trusts_price_history_monthly.csv` — monthly, price only, no
dividends, and every ticker starts 2011-09.

**Yahoo has replaced this as the price source**, with deeper history and dividends. The
file is still landed for one reason: Yahoo deletes delisted symbols, so the trusts that
died are invisible to it. This CSV is the only record of them this project holds.

Silver will take two tickers from here — `BCPT` (2011-09 to 2024-11) and `CSH` (2016-12
to 2026-03). Everything else in the file is superseded.

Lands exactly as it arrived, known problems included: date labels mix month-end and
next-month-first, ~25 tickers carry interleaved rows on a wrong scale, and `PCFT` has a
zero price. Silver handles those.

Expected: **16,357 rows**.

In [0]:
import os

import pandas as pd

CATALOG = "`index-vs-trust-pipeline`"
TABLE = f"{CATALOG}.landing.trust_prices_csv_raw"

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CSV_PATH = os.path.join(
    REPO_ROOT, "data", "uk_investment_trusts_price_history_monthly.csv"
)

print(f"reading {CSV_PATH}")

In [0]:
# Text in, text out -- the price column stays a string so a malformed number survives
# to Silver instead of becoming a null at the front door.
prices = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

print(f"{len(prices)} rows, {len(prices.columns)} columns")
print(list(prices.columns))
prices.head(3)

In [0]:
sdf = spark.createDataFrame(prices)
sdf.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print(f"wrote {TABLE}")

## Verification

In [0]:
%sql
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT ticker) AS distinct_tickers,
       MIN(`date`) AS first_date,
       MAX(`date`) AS last_date
FROM `index-vs-trust-pipeline`.landing.trust_prices_csv_raw;

Expect **16,357 rows, 102 tickers, 2011-09-30 to 2026-09-17**.

In [0]:
%sql
-- The known-bad row must still be bad. This proves Landing cleaned nothing.
SELECT ticker, `date`, price_gbx_or_gbp
FROM `index-vs-trust-pipeline`.landing.trust_prices_csv_raw
WHERE ticker = 'PCFT' AND `date` = '2019-11-01';

In [0]:
%sql
-- The two tickers this table exists for. Yahoo returns nothing for either.
SELECT ticker,
       COUNT(*)    AS rows_held,
       MIN(`date`) AS first_date,
       MAX(`date`) AS last_date
FROM `index-vs-trust-pipeline`.landing.trust_prices_csv_raw
WHERE ticker IN ('BCPT', 'CSH')
GROUP BY ticker
ORDER BY ticker;

Expect **`BCPT` 158 rows to 2024-11-01** and **`CSH` 112 rows to 2026-03-01**. These two
are the entire delisted cohort, and the whole reason this table is still built.